In [1]:
import json

In [2]:
import glob
for ifs_f in glob.glob("ifs_*.json"):
    curr_f = json.load(open(ifs_f))
    print(ifs_f)
    all_ifs_raw = [curr_f[i]["ifs_raw"] for i in curr_f]
    all_ifs_normalized = [curr_f[i]["ifs_normalized"] for i in curr_f]
    print(sum(all_ifs_raw) / float(len(all_ifs_raw)))
    print(sum(all_ifs_normalized) / float(len(all_ifs_normalized)))

ifs_PAPILLON_Qwen_Qwen3-4B-Instruct-2507_2026-03-26_15:19:26.json.json
17.68777754113436
0.17269536298378646
ifs_PAPILLON_Qwen_Qwen3-8B_2026-03-26_15:48:52.json.json
27.935676619116787
0.36723873243209654
ifs_GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_qwen_15b_inst_full_2026-03-29_17:39:14.json.json
3.9978034987516784
0.003896524216498271
ifs_GSM8k_Qwen_Qwen2.5-7B-Instruct_2026-03-24_19:08:46.json.json
2.965776554444358
0.04021323259025638
ifs_PAPILLON_Qwen_Qwen2.5-7B-Instruct_2026-03-26_14:43:00.json.json
16.621419813566277
0.19670198390025384
ifs_GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json.json
3.942034666595396
0.028927133621020524
ifs_GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_debug_2026-03-27_17:12:18.json.json
4.156084008054275
0.0320940690200747
ifs_GSM8k_Qwen_Qwen3-4B-Instruct-2507_2026-03-24_20:38:23.json.json
4.710243594422272
0.01684505549321167
ifs_PAPILLON_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.j

In [8]:
# gsm_traces = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json"))
gsm_traces = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_qwen_15b_inst_full_2026-03-29_17:39:14.json"))

In [9]:
import os
import re
from scipy.stats import spearmanr


REP_STR = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant"

def extract_final_answer(completion: str):
    """Extract the number after 'The final answer is' from a completion string."""
    completion = completion.replace(",", "")
    match = re.findall(r"\d+", completion)
    if match:
        return match[-1].strip()
    return None


for f in glob.glob("GSM8k_*.json"):
    if os.path.exists("ifs_" + f + ".json"):
        print(f)
        gsm_traces = json.load(open(f))
        ifs_corr = []
        num_corr = []
        ifs_scores = json.load(open("ifs_" + f + ".json"))
        for i, group in enumerate(gsm_traces):
            comps = [c.replace(REP_STR, "") for c in group["comps"]]
            answers = [extract_final_answer(c) for c in comps]
            non_null = [a for a in answers if a is not None]

            if not non_null:
                continue

            unique_answers = set(non_null)
            if str(i) in ifs_scores:
                ifs_corr.append(ifs_scores[str(i)]["ifs_raw"])
                num_corr.append(len(unique_answers))
        print(spearmanr(ifs_corr, num_corr))

GSM8k_Qwen_Qwen3-4B-Instruct-2507_2026-03-24_20:38:23.json
SignificanceResult(statistic=np.float64(0.30641872860277936), pvalue=np.float64(5.7921576938719735e-08))
GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_qwen_15b_inst_full_2026-03-29_17:39:14.json
SignificanceResult(statistic=np.float64(0.14816780512458708), pvalue=np.float64(0.5215443811079751))
GSM8k_Qwen_Qwen2.5-7B-Instruct_2026-03-24_19:08:46.json
SignificanceResult(statistic=np.float64(0.18366725688352942), pvalue=np.float64(0.0013718041054025859))
GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_debug_2026-03-27_17:12:18.json
SignificanceResult(statistic=np.float64(0.5095204765591677), pvalue=np.float64(2.734077866209343e-21))
GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json
SignificanceResult(statistic=np.float64(0.5123798409228931), pvalue=np.float64(1.5055480156734975e-21))


In [9]:
papillon_f = "/local-storage/interaction/siyanli/DP_PAPILLON/papillon/PAPILLON_Qwen_Qwen2.5-7B-Instruct_2026-03-26_14:43:00.json"
papillon_traces = json.load(open(papillon_f))

In [13]:
papillon_traces[0]["comps"][0]

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n[[ ## createdPrompt ## ]]\nCreate a friendly email to a colleague, expressing gratitude for their prompt response and detailed discussion about a company program. Mention their willingness to make accommodations to ensure equal access to the program for individuals with disabilities. Highlight the positive impact their kindness has had.\n\n[[ ## completed ## ]]<|im_end|>\n'

In [32]:
import re
from task_customizer import PAPILLONCustomizer
from constants import MALE_NAME_LIST, FEMALE_NAME_LIST

def extract_user_query(s):
    pattern = r'user\n\[\[ ## userQuery ## \]\]\n(.*?)\n\nRespond with'
    match = re.search(pattern, s, re.DOTALL)
    if match:
        user_query = match.group(1)
    return user_query

def extract_created_prompt(s):
    pattern = r'\[\[ ## createdPrompt ## \]\](.*?)\[\[ ## completed ## \]\]'
    match = re.search(pattern, s, re.DOTALL)
    if match:
        created_prompt = match.group(1)
    else:
        return None
    return created_prompt

def check_pii_presence(piis, s):
    num_pii = 0
    for p in piis.split("||") + MALE_NAME_LIST + FEMALE_NAME_LIST:
        if p.lower() in s.lower():
            num_pii += 1
    return num_pii / len(piis.split("||"))

papillon_customizer = PAPILLONCustomizer()


for papillon_f in glob.glob("PAPILLON*.json"):
    if os.path.exists("ifs_" + papillon_f + ".json"):
        papillon_traces = json.load(open(papillon_f))
        papillon_scores = json.load(open("ifs_" + papillon_f + ".json"))
        all_pcts, all_scores = [], []
        all_rdds, all_ind_scores = [], []
        for i in range(len(papillon_traces)):
            curr_og_prompt = extract_user_query(papillon_traces[i]["prompts"][0])
            curr_pii = papillon_customizer.data_pii_dict[curr_og_prompt]
            curr_pii_pcts = []
            for j, c in enumerate(papillon_traces[i]["comps"]):
                curr_created_prompt = extract_created_prompt(c)
                if curr_created_prompt:
                    curr_pii_pcts.append(check_pii_presence(curr_pii, curr_created_prompt))
                    all_rdds.append(papillon_scores[str(i)]["rdd_scores"][j])
                    all_ind_scores.append(check_pii_presence(curr_pii, curr_created_prompt))
            if len(curr_pii_pcts):
                all_pcts.append(sum(curr_pii_pcts) / len(curr_pii_pcts))
                all_scores.append(papillon_scores[str(i)]["ifs_raw"])
        print(papillon_f)
        print(max(all_pcts), min(all_pcts))
        print(spearmanr(all_pcts, all_scores))
        print(spearmanr(all_rdds, all_ind_scores))
            




PAPILLON Customizer Data Size: 184
PAPILLON_Qwen_Qwen3-0.6B_2026-03-26_16:40:37.json
1.9642857142857142 0.0
SignificanceResult(statistic=np.float64(0.4003653682274739), pvalue=np.float64(1.7902324990392428e-08))
SignificanceResult(statistic=np.float64(0.30414357131175407), pvalue=np.float64(2.8190427527130005e-79))
PAPILLON_Qwen_Qwen2.5-7B-Instruct_2026-03-26_14:43:00.json
1.967741935483871 0.0
SignificanceResult(statistic=np.float64(0.806903891186324), pvalue=np.float64(1.7770943772543826e-43))
SignificanceResult(statistic=np.float64(0.5340063727608289), pvalue=np.float64(1.146382830584841e-270))
PAPILLON_Qwen_Qwen3-8B_2026-03-26_15:48:52.json
1.64 0.0
SignificanceResult(statistic=np.float64(0.5601063814184797), pvalue=np.float64(2.4459834235799737e-16))
SignificanceResult(statistic=np.float64(0.1370730924527824), pvalue=np.float64(4.138397022948344e-16))
PAPILLON_Qwen_Qwen3-4B-Instruct-2507_2026-03-26_15:19:26.json
1.9615384615384615 0.0
SignificanceResult(statistic=np.float64(0.8332

In [10]:
import re

REP_STR = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant"

def extract_final_answer(completion: str):
    """Extract the number after 'The final answer is' from a completion string."""
    completion = completion.replace(",", "")
    match = re.findall(r"\d+", completion)
    if match:
        return match[-1].strip()
    return None

# For each group, extract answers and check if all variants agree
all_same, all_none, groups_with_disagreement = [], [], []

# ifs_scores = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/ifs_GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json.json"))
ifs_scores = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/ifs_GSM8k__local-storage_interaction_siyanli_DP_PAPILLON_papillon_grpo_ifs_qwen_15b_inst_full_2026-03-29_17:39:14.json.json"))
ifs_corr = []
num_corr = []

for i, group in enumerate(gsm_traces):
    comps = [c.replace(REP_STR, "") for c in group["comps"]]
    answers = [extract_final_answer(c) for c in comps]
    non_null = [a for a in answers if a is not None]

    if not non_null:
        all_none.append(i)
        continue

    unique_answers = set(non_null)
    if len(unique_answers) == 1 and len(non_null) == len(answers):
        all_same.append(i)
    else:
        groups_with_disagreement.append({
            "group": i,
            "answers": answers,
            "unique": unique_answers,
        })
    if str(i) in ifs_scores:
        ifs_corr.append(ifs_scores[str(i)]["ifs_normalized"])
        num_corr.append(len(unique_answers))


print(f"Total groups          : {len(gsm_traces)}")
print(f"All variants agree    : {len(all_same)} ({100*len(all_same)/len(gsm_traces):.1f}%)")
print(f"Disagreement          : {len(groups_with_disagreement)} ({100*len(groups_with_disagreement)/len(gsm_traces):.1f}%)")
print(f"No answer extracted   : {len(all_none)} ({100*len(all_none)/len(gsm_traces):.1f}%)")


Total groups          : 1180
All variants agree    : 709 (60.1%)
Disagreement          : 471 (39.9%)
No answer extracted   : 0 (0.0%)


In [11]:
from scipy.stats import spearmanr
spearmanr(ifs_corr, num_corr)

SignificanceResult(statistic=np.float64(-0.1461654024219543), pvalue=np.float64(0.5163007182880918))

In [12]:
# Inspect a few disagreement cases
for g in groups_with_disagreement[:5]:
    i = g["group"]
    print(f"Group {i} | unique answers: {g['unique']}")
    for variant_idx, (prompt, answer) in enumerate(zip(gsm_traces[i]["prompts"], g["answers"])):
        print(f"  variant {variant_idx}: answer={answer}")
    print()


Group 2 | unique answers: {'55', '5'}
  variant 0: answer=55
  variant 1: answer=5
  variant 2: answer=5
  variant 3: answer=5
  variant 4: answer=5
  variant 5: answer=5
  variant 6: answer=5
  variant 7: answer=5
  variant 8: answer=5
  variant 9: answer=5
  variant 10: answer=5
  variant 11: answer=5
  variant 12: answer=5
  variant 13: answer=5
  variant 14: answer=5
  variant 15: answer=5

Group 4 | unique answers: {'624', '312'}
  variant 0: answer=624
  variant 1: answer=624
  variant 2: answer=312
  variant 3: answer=312
  variant 4: answer=624
  variant 5: answer=312
  variant 6: answer=624
  variant 7: answer=624
  variant 8: answer=624
  variant 9: answer=624
  variant 10: answer=624
  variant 11: answer=624
  variant 12: answer=312
  variant 13: answer=624
  variant 14: answer=312
  variant 15: answer=624

Group 5 | unique answers: {'22', '35'}
  variant 0: answer=35
  variant 1: answer=22
  variant 2: answer=35
  variant 3: answer=35
  variant 4: answer=22
  variant 5: ans

In [8]:
from constants import MALE_NAME_LIST, FEMALE_NAME_LIST

name_cts, total_cts = 0, 0

for i, group in enumerate(gsm_traces):
    comps = [c.replace(REP_STR, "") for c in group["comps"]]
    for c in comps:
        for n in MALE_NAME_LIST + FEMALE_NAME_LIST:
            if n in c:
                name_cts += 1
                break
        total_cts += 1
    total_cts -= 1
print(name_cts / float(total_cts))

0.9140677966101695
